# Evaluation Notebook (GPTQ)

This notebook evaluates the `gptq_w4` model and saves the results into `quantized_models/gptq_w4/`.

It can also evaluate the baseline FP16 model for a direct comparison.

> **Note:** Run this notebook in the Conda environment that supports `auto-gptq` (e.g., `gptq-env`).

In [1]:
# # Install any missing packages (run once)
# # You can comment this cell out once dependencies are satisfied.

# import sys
# import subprocess

# REQUIRED = [
#     "torch",
#     "transformers",
#     "datasets",
#     "accelerate",
#     "auto-gptq",
#     "pandas",
#     "tqdm",
# ]

# for pkg in REQUIRED:
#     try:
#         __import__(pkg)
#     except ImportError:
#         subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# print("Dependencies verified.")

In [2]:
from pathlib import Path

import torch
from transformers import AutoTokenizer

import utils.model_loader as model_loader
import utils.eval_utils as eval_utils

# Where quantized models are stored
OUTPUT_ROOT = Path("./quantized_models")

# Configuration: list of models to evaluate
MODEL_CONFIGS = [{
    "name": "gptq_w4", 
    "path": str(OUTPUT_ROOT / "gptq_w4"), 
    "bits": 4,  
    "run_lm_harness": True
}]

# Perplexity evaluation settings
PERPLEXITY_SAMPLES = 100
MAX_LENGTH = 512

In [3]:
def load_and_prepare_model(config: dict, device: str = "cuda"):
    """Load a model and tokenizer for evaluation."""

    model_path = config["path"]
    model_name = config["name"]

    if "gptq" in model_name.lower():
        from auto_gptq import AutoGPTQForCausalLM

        model = AutoGPTQForCausalLM.from_quantized(
            model_path,
            device="cuda:0" if torch.cuda.is_available() else "cpu",
            use_safetensors=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)

    else:
        model = model_loader.load_base_model(model_path)
        tokenizer = model_loader.load_tokenizer(model_path)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def evaluate_model_config(config: dict):
    """Evaluate a model and save the results into its model directory."""

    print("\n" + "=" * 80)
    print(f"Evaluating: {config['name']}")
    print("=" * 80)

    model_dir = config["path"]
    model_name = config["name"]
    # model_dir.mkdir(parents=True, exist_ok=True)

    model, tokenizer = load_and_prepare_model(config)

    size_metrics = eval_utils.calculate_model_size_and_bandwidth(
        model, bits=config.get("bits", 16)
    )

    ppl_metrics = eval_utils.evaluate_perplexity(
        model,
        tokenizer,
        max_samples=PERPLEXITY_SAMPLES,
        max_length=MAX_LENGTH,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

    lm_metrics = {}
    if config.get("run_lm_harness", False):
        if "gptq" in model_name.lower():
            model = getattr(model, "model", model)

        lm_metrics = eval_utils.evaluate_lm_harness(
            model_obj=model,
            tokenizer_obj=tokenizer,
            # Updated tasks to specific MMLU groups
            tasks=["hellaswag", "piqa", "mmlu_stem", "mmlu_humanities"],
            num_fewshot=3,
            limit=5,
        )

    results = {
        "model_name": config["name"],
        "model_path": str(model_dir),
        "bits": config.get("bits", 16),
        **size_metrics,
        **ppl_metrics,
        "lm_eval": lm_metrics
    }

    eval_utils.save_eval_result(results, model_dir, file_stem="evaluation")

    del model
    torch.cuda.empty_cache()

    print(f"Saved evaluation to: evaluation.json")
    return results


# Run evaluation for all configured models
all_results = []
for cfg in MODEL_CONFIGS:
    try:
        results = evaluate_model_config(cfg)
        all_results.append(results)
    except Exception as e:
        print(f"Failed to evaluate {cfg['name']}: {e}")

# Save combined results (optional)
combined_dir = OUTPUT_ROOT / "analysis"
combined_dir.mkdir(parents=True, exist_ok=True)

import pandas as pd

if all_results:
    df = pd.DataFrame(all_results)
    df.to_csv(combined_dir / "evaluation_gptq_results.csv", index=False)
    df.to_json(combined_dir / "evaluation_gptq_results.json", orient="records", indent=2)
    print(f"\nSaved combined results to: {combined_dir}")


Evaluating: gptq_w4


/mnt/miniconda3/envs/gptq-env/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/mnt/miniconda3/envs/gptq-env/lib/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:360: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, qweight, scales, qzeros, g_idx, bits, maxq):
/mnt/miniconda3/envs/gptq-env/lib/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:368: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/mnt/miniconda3/envs/gptq-env/lib/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:399: FutureWarning:

Evaluating perplexity on wikitext test split (100 samples)...


Evaluating: 100%|██████████| 100/100 [00:10<00:00,  9.15it/s]
`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
Overwriting default num_fewshot of mmlu_formal_logic from None to 3
Overwriting default num_fewshot of mmlu_high_school_european_history from None to 3
Overwriting default num_fewshot of mmlu_high_school_us_history from None to 3
Overwriting default num_fewshot of mmlu_high_school_world_history from None to 3
Overwriting default num_fewshot of mmlu_international_law from None to 3
Overwriting default num_fewshot of mmlu_jurisprudence from None to 3
Overwriting default num_fewshot of mmlu_logical_fallacies from None to 3
Overwriting default num_fewshot of mmlu_moral_disputes from None to 3
Overwriting defa

Saved evaluation to: evaluation.json

Saved combined results to: quantized_models/analysis
